# Model comparison

## have a baseline, use imagehash downsampling and augmentation, now very model creations but need better compiling

In [ ]:
import numpy as np
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import os
from PIL import Image
import imagehash
import imageio.v3 as imageio
import pandas as pd
import seaborn as sns
from collections import Counter
import random
import keras_tuner as kt


Image_size = 96
Batch = 32
Random_seed = 42

# quick runthrough of data notebook, using image hash and augmentation

In [ ]:
def get_clean_data_ImageHash(dirs):
    all_paths, all_labels = [], []
    for d in dirs:

        # Finding folder paths 
        class_names = sorted(os.listdir(d))
        class_to_idx = {name: i for i, name in enumerate(class_names)}
        for class_name in class_names:
            c_path = os.path.join(d, class_name)
            if not os.path.isdir(c_path): continue
            files = [os.path.join(c_path, f) for f in os.listdir(c_path)]
            all_paths.extend(files)
            all_labels.extend([class_to_idx[class_name]] * len(files))
    
    # Use hashing to remove duplicates in all folders
    unique_hashes = {}
    clean_paths, clean_labels = [], []
    print(f"Total raw files: {len(all_paths)}")
    for p, l in zip(all_paths, all_labels):
        try:
            with Image.open(p) as img:
                h = str(imagehash.phash(img))
                if h not in unique_hashes:
                    unique_hashes[h] = p
                    clean_paths.append(p)
                    clean_labels.append(l)
        except: continue
    print(f"Total unique files: {len(clean_paths)}")
    return clean_paths, clean_labels


def balance_split(paths, labels):
    counts = Counter(labels)
    min_count = min(counts.values())
    combined = list(zip(paths, labels))
    random.shuffle(combined)
    bal_p, bal_l = [], []
    class_seen = Counter()
    for p, l in combined:
        if class_seen[l] < min_count:
            bal_p.append(p)
            bal_l.append(l)
            class_seen[l] += 1
    return bal_p, bal_l

def augment(image, label):
    # Various augmentations, from rotation to random saturation and blurring
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform((), 0, 4, dtype=tf.int32))
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    image = tf.image.random_hue(image, max_delta=0.05)
    
    # Haze - fixed with tf.cond
    haze_intensity = tf.random.uniform((), 0.05, 0.25)
    haze = tf.ones_like(image) * 0.7
    image = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: image * (1 - haze_intensity) + haze * haze_intensity,
        lambda: image
    )

    # Blur - fixed with tf.cond
    def apply_blur(img):
        img = tf.expand_dims(img, 0)
        img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
        return tf.squeeze(img, 0)

    image = tf.cond(
        tf.random.uniform(()) > 0.6,
        lambda: apply_blur(image),
        lambda: image
    )

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label


def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [Image_size, Image_size])
    return img, label

def prepare_dataset(paths, labels, use_aug=False, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths))
    ds = ds.map(load_and_preprocess)
    if use_aug:
        ds = ds.map(augment)
    ds = ds.map(normalize).batch(Batch).prefetch(tf.data.AUTOTUNE)
    return ds

all_p, all_l = get_clean_data_ImageHash(["Training/", "Test/"])

tr_p, temp_p, tr_l, temp_l = train_test_split(
    all_p, all_l, test_size=0.25, random_state=Random_seed, stratify=all_l
)
val_p, te_p, val_l, te_l = train_test_split(
    temp_p, temp_l, test_size=0.5, random_state=Random_seed, stratify=temp_l
)

In [ ]:
train_ds = (tf.data.Dataset.from_tensor_slices((tr_p, tr_l))
            .shuffle(len(tr_p)).map(load_and_preprocess).map(augment).batch(Batch).prefetch(2))
val_ds = (tf.data.Dataset.from_tensor_slices((val_p, val_l))
          .map(load_and_preprocess).map(normalize).batch(Batch).prefetch(2))
test_ds = (tf.data.Dataset.from_tensor_slices((te_p, te_l))
           .map(load_and_preprocess).map(normalize).batch(Batch).prefetch(2))


weights = compute_class_weight('balanced', classes=np.unique(tr_l), y=tr_l)
class_weights = dict(enumerate(weights))

# Train on a a complex model, layers with normal compiling metrics

In [ ]:
model = keras.models.Sequential([
    keras.layers.Conv2D(4, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(4, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.1),

    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
    # Block 1
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
    # Block 2
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.3),
    # Block 3
    keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.4),
    # Block 4
    keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    keras.layers.GlobalAveragePooling2D(),
    # Head
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss = 'binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
    ]
)
history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    class_weight=class_weights,
)

In [ ]:
loss, acc, recall, precision = model.evaluate(test_ds)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test Precision: {precision:.4f}")

# Get probabilities
y_pred_prob = model.predict(test_ds)

true_labels = np.concatenate([y.numpy() for _, y in test_ds])

pred_classes = (y_pred_prob > 0.50).astype(int).flatten()
cm = confusion_matrix(true_labels, pred_classes)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(true_labels, pred_classes, target_names=['No_Fire', 'Fire']))

# Its very large and has room for improvement. make it over fitt. i will compare to smaller models to see if its better .



In [ ]:
def build_binary_model_scaled(model_size='medium', loss_fn=None):
    scales = {
        'very_small': (4, 8, 8),   
        'small':      (8, 16, 16),  
        'medium':     (8, 16, 16 ,32),  
        'large':      (8, 8, 16, 16, 32, 64), 
    }
    num_blocks, base_filters, dense_dim = scales[model_size]
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(96, 96, 3)))

    for i in range(num_blocks):
        filters = base_filters * (2**i)
        #model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.MaxPooling2D())
        model.add(keras.layers.Dropout(0.2 if i < 2 else 0.3))

    model.add(keras.layers.Conv2D(filters * 2, 3, activation='relu', padding='same'))
    model.add(keras.layers.GlobalAveragePooling2D())
    model.add(keras.layers.Dense(dense_dim, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(1, activation='sigmoid')) 

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss = 'binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
    )
    return model

def run_binary_experiment(size, loss_func, weights, callbacks):
    print(f"\n--- TESTING BINARY {size.upper()} ---")
    model = build_binary_model_scaled(size, loss_func)
    history = model.fit(train_ds, validation_data=val_ds, epochs=30, 
                        class_weight=weights, callbacks=callbacks, verbose=1)
    
    metrics = ['loss', 'accuracy', 'precision', 'recall']
    plt.figure(figsize=(20, 5))
    for i, metric in enumerate(metrics):
        plt.subplot(1, 4, i+1)
        plt.plot(history.history[metric], label='Train')
        plt.plot(history.history[f'val_{metric}'], label='Val')
        plt.title(f'{size} {metric}')
        plt.legend()
    plt.show()
    return model, history

In [ ]:

def plot_model_comparison(results_dict):
    data = []
    for size, result in results_dict.items():
        if isinstance(result, tuple):
            history_obj = result[1]
        else:
            history_obj = result
            
        h_dict = history_obj.history if hasattr(history_obj, 'history') else history_obj
        
        best_epoch = np.argmin(h_dict['val_loss'])
        
        row = {
            'Size': size,
            'Val Accuracy': h_dict['val_accuracy'][best_epoch],
            'Val Loss': h_dict['val_loss'][best_epoch],
        }
        
        if 'val_precision' in h_dict:
            row['Val Precision'] = h_dict['val_precision'][best_epoch]
        if 'val_recall' in h_dict:
            row['Val Recall'] = h_dict['val_recall'][best_epoch]
            
        data.append(row)

    df = pd.DataFrame(data)
    
    # ADD 'EfficientNetV2' TO THIS LIST:
    order = ['very_small', 'small', 'medium', 'large', 'EfficientNetV2']
    
    # Filter order to only include sizes actually present in the results
    actual_order = [s for s in order if s in df['Size'].values]
    
    # This line was turning 'EfficientNetV2' into NaN because it wasn't in actual_order
    df['Size'] = pd.Categorical(df['Size'], categories=actual_order, ordered=True)
    df = df.sort_values('Size')
    df_melted = df.melt(id_vars='Size', var_name='Metric', value_name='Value')
    df_metrics = df_melted[df_melted['Metric'] != 'Val Loss']
    df_loss = df_melted[df_melted['Metric'] == 'Val Loss']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    sns.barplot(data=df_metrics, x='Size', y='Value', hue='Metric', ax=ax1, palette='magma')
    ax1.set_title('Performance Metrics Comparison', fontsize=15, fontweight='bold')
    ax1.set_ylim(0, 1.05)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    sns.lineplot(data=df_loss, x='Size', y='Value', marker='o', ax=ax2, color='crimson', linewidth=3, markersize=10)
    ax2.set_title('Validation Loss (Lower is Better)', fontsize=15, fontweight='bold')
    ax2.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()
plot_model_comparison(all_results)

# compare with focal loss and binary and using weigh

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return tf.reduce_mean(alpha * tf.pow(1 - p_t, gamma) * bce)
    return loss

In [ ]:
def build_binary_model_scaled_adv(model_size='medium', loss_fn=None):
    scales = {
        'very_small': (4, 8, 8),   
        'small':      (8, 16, 16),  
        'medium':     (8, 16, 16 ,32),  
        'large':      (8, 8, 16, 16, 32, 64), 
    }
    num_blocks, base_filters, dense_dim = scales[model_size]
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(96, 96, 3)))

    for i in range(num_blocks):
        filters = base_filters * (2**i)
        #model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.MaxPooling2D())
        model.add(keras.layers.Dropout(0.2 if i < 2 else 0.3))

    model.add(keras.layers.Conv2D(filters * 2, 3, activation='relu', padding='same'))
    model.add(keras.layers.GlobalAveragePooling2D())
    model.add(keras.layers.Dense(dense_dim, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(1, activation='sigmoid')) 

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss = focal_loss(gamma=2.0, alpha=0.25) if loss_fn == 'focal' else 'binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
    )
    return model

def run_binary_experiment(size, loss_func, weights, callbacks):
    print(f"\n--- TESTING BINARY {size.upper()} ---")
    model = build_binary_model_scaled_adv(size, loss_func)
    history = model.fit(train_ds, validation_data=val_ds, epochs=30, 
                        class_weight=weights, callbacks=callbacks, verbose=1)
    
    metrics = ['loss', 'accuracy', 'precision', 'recall']
    plt.figure(figsize=(20, 5))
    for i, metric in enumerate(metrics):
        plt.subplot(1, 4, i+1)
        plt.plot(history.history[metric], label='Train')
        plt.plot(history.history[f'val_{metric}'], label='Val')
        plt.title(f'{size} {metric}')
        plt.legend()
    plt.show()
    return model, history

In [ ]:
my_loss = focal_loss(gamma=2.0, alpha=0.25)
my_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

all_results = {}
for model_size in ['very_small','small', 'medium', 'large']:
    all_results[model_size] = run_binary_experiment(model_size, my_loss, class_weights, my_callbacks)

# Now i compare it to standard models in this field, efficientnetV2 same layers as best performing model

In [ ]:
def build_effnet_binary_model(loss_fn=None):
    inputs = keras.layers.Input(shape=(96, 96, 3))
    
    # Scale [0, 1] back up to [0, 255] for EfficientNetV2's internal preprocessing
    x = keras.layers.Lambda(lambda tensor: tensor * 255.0)(inputs)

    # Load pre-trained EfficientNetV2B0 without the classification head
    base_model = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_tensor=x,
        input_shape=(96, 96, 3)
    )
    
    # leave the base model trainable to fine-tune it on specific fire data
    base_model.trainable = True

    # Rebuild the top layers
    x = base_model.output
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid')(x)

    model = keras.models.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=loss_fn if loss_fn else 'binary_crossentropy',
        metrics=[
            'accuracy', 
            keras.metrics.Precision(name='precision'), 
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

print("\n--- TESTING EFFICIENTNETV2B0 (BINARY) ---")
effnet_bin_model = build_effnet_binary_model(my_loss)

history_effnet_bin = effnet_bin_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=my_callbacks,
    verbose=1
)

# Append to your existing results dictionary
all_results['EfficientNetV2'] = (effnet_bin_model, history_effnet_bin)

# Now, re-run your plot function to see the comparison!
plot_model_comparison(all_results)

# can use hyperparameter tuner to find best model hyperparameters

In [ ]:
def build_tuning_model(hp):
    
    base_filters = 8

    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(96, 96, 3)))

    for i in range(2): 
        filters = base_filters * (2**i)
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.MaxPooling2D())
        
        hp_dropout_conv = hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.3, step=0.1)
        model.add(keras.layers.Dropout(hp_dropout_conv))

    model.add(keras.layers.Conv2D(filters * 2, 3, activation='relu', padding='same'))
    model.add(keras.layers.GlobalAveragePooling2D())
    
    hp_dense_units = hp.Int('dense_units', min_value=16, max_value=64, step=16)
    model.add(keras.layers.Dense(hp_dense_units, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(1, activation='sigmoid')) 

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss=focal_loss(gamma=2.0, alpha=0.25),
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
    )
    return model

# 3. Instantiate the Hyperband tuner
tuner = kt.Hyperband(
    build_tuning_model,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory='hyperband_small_model',
    project_name='fire_small_tuning'
)

# 4. Search for the best hyperparameters
stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[stop_early],
    class_weight=class_weights # From your notebook
)

# 5. Get and train the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Dense Units: {best_hps.get('dense_units')}")
print(f"Best Learning Rate: {best_hps.get('learning_rate')}")

final_model = tuner.hypermodel.build(best_hps)
history = final_model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=30, 
    class_weight=class_weights
)

# model type, conv vs seperable conv

In [ ]:
def build_small_comparison(use_separable=False):
    # Based on your 'small' config: (8, 16, 16)
    # 2 blocks of base_filters, then 1 final conv of filters * 2
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(96, 96, 3)))
    
    # Choose layer type
    LayerType = keras.layers.SeparableConv2D if use_separable else keras.layers.Conv2D
    
    # Block 1 (filters=8)
    model.add(LayerType(8, 3, activation='relu', padding='same'))
    model.add(keras.layers.MaxPooling2D())
    model.add(keras.layers.Dropout(0.2))
    
    # Block 2 (filters=16)
    model.add(LayerType(16, 3, activation='relu', padding='same'))
    model.add(keras.layers.MaxPooling2D())
    model.add(keras.layers.Dropout(0.2))

    # Final Conv (filters=16 * 2)
    model.add(LayerType(32, 3, activation='relu', padding='same'))
    model.add(keras.layers.GlobalAveragePooling2D())
    model.add(keras.layers.Dense(16, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(1, activation='sigmoid')) 

    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Create both versions
std_model = build_small_comparison(use_separable=False)
sep_model = build_small_comparison(use_separable=True)

# Comparison Output
print(f"{'Model Type':<20} | {'Total Parameters':<20}")
print("-" * 45)
print(f"{'Standard Conv2D':<20} | {std_model.count_params():<20,}")
print(f"{'Separable Conv2D':<20} | {sep_model.count_params():<20,}")

reduction = (1 - sep_model.count_params() / std_model.count_params()) * 100
print(f"\nParameter Reduction: {reduction:.2f}%")